<a href="https://colab.research.google.com/github/ksafford05/capstone_prescribed_burn/blob/main/scripts/1_data_cleaning_capstone.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# **DATA CLEANING SCRIPT** for Longleaf Pine

---



---



In [1]:
#mount google drive
from google.colab import drive

In [1]:
#change runtime back to R
#load libraries
library(readr)
library(dplyr)
library(tidyr)



Attaching package: ‘dplyr’


The following objects are masked from ‘package:stats’:

    filter, lag


The following objects are masked from ‘package:base’:

    intersect, setdiff, setequal, union




In [2]:
#set working directory as capstone folder
setwd("/content/drive/MyDrive/YALE_Enviro_Data_Science/capstone_prescribed_burn")
getwd()

[1] "/content/drive/MyDrive/YALE_Enviro_Data_Science/capstone_prescribed_burn"

In [3]:
#read in raw data
sc_plot     <- read_csv("raw_data/SC_PLOT.csv") #Location and survey years
sc_type     <- read_csv("raw_data/SC_COND.csv") #Forest type and fire history
sc_tree     <- read_csv("raw_data/SC_TREE.csv") #Overstory and midstory competition, filter for longleaf pine
sc_seedling <- read_csv("raw_data/SC_SEEDLING.csv") #regeneration data


Rows: 45675 Columns: 63
── Column specification ────────────────────────────────────────────────────────
Delimiter: ","
chr   (1): MICROPLOT_LOC
dbl  (40): CN, SRV_CN, CTY_CN, PREV_PLT_CN, INVYR, STATECD, UNITCD, COUNTYCD...
lgl  (20): KINDCD_NC, DECLINATION, MACRO_BREAKPOINT_DIA, TOPO_POSITION_PNW, ...
dttm  (2): CREATED_DATE, MODIFIED_DATE

ℹ Use `spec()` to retrieve the full column specification for this data.
ℹ Specify the column types or set `show_col_types = FALSE` to quiet this message.
Warning message:
“One or more parsing issues, call `problems()` on your data frame for details,
e.g.:
  dat <- vroom(...)
  problems(dat)”
Rows: 54684 Columns: 153
── Column specification ────────────────────────────────────────────────────────
Delimiter: ","
chr  (11): PROP_BASIS, VOL_LOC_GRP, NVCS_PRIMARY_CLASS, NVCS_LEVEL_2_CD, NVC...
dbl  (81): CN, PLT_CN, INVYR, STATECD, UNITCD, COUNTYCD, PLOT, CONDID, COND_...
lgl  (59): MACRPROP_UNADJ, DSTRBYR3, HABTYPCD1, HABTYPCD1_PUB_CD, HABTYPCD1_...
d

In [4]:
# First dataset: filter SC_Plot to Berkley (ID == 15) and Charleston Counties (ID==19)and 2005+, Join CN in the Plot table by PLT_CN
clean_sc_plot <- sc_plot %>%
  filter(
   COUNTYCD %in% c(15,19), #filtering for 2 counties
   INVYR >= 2005    #filtering to match records from other datasets
  ) %>%
select( #select columns needed
  COUNTYCD,
  INVYR,
  CN,
  PLOT
) %>%
rename( #rename variables
  countyid      = COUNTYCD,
  inventory_yr  =  INVYR,
)

glimpse(clean_sc_plot)
head(clean_sc_plot,10)

Rows: 1,348
Columns: 4
$ countyid     <dbl> 15, 15, 15, 15, 15, 15, 15, 15, 15, 15, 19, 19, 19, 19, 1…
$ inventory_yr <dbl> 2005, 2005, 2005, 2005, 2005, 2005, 2005, 2005, 2005, 200…
$ CN           <dbl> 2.407165e+13, 2.407178e+13, 2.407180e+13, 2.407181e+13, 2…
$ PLOT         <dbl> 41, 16, 12, 5, 117, 127, 137, 145, 112, 1, 122, 24, 4, 11…


countyid,inventory_yr,CN,PLOT
<dbl>,<dbl>,<dbl>,<dbl>
15,2005,2.407165e+13,41
15,2005,2.407178e+13,16
15,2005,2.407180e+13,12
15,2005,2.407181e+13,5
15,2005,2.407209e+13,117
15,2005,2.407215e+13,127
15,2005,2.407223e+13,137
15,2005,2.407224e+13,145
15,2005,2.407171e+13,112


In [22]:
#Second dataset: filter sc_type to only include National Forest Land (#10) and longleaf pine (#141)
clean_sc_type <- sc_type %>%
  filter(
    OWNGRPCD == 10,
    FORTYPCD == 141,
    COUNTYCD %in% c(15, 19),
    INVYR >= 2005
  ) %>%
  select(PLT_CN, FORTYPCD, STDAGE)%>% #select columns needed
  rename( #rename columns, match to first dataset
    stand_age_yrs  = STDAGE,
    forest_type    = FORTYPCD
)
head(clean_sc_type, 15)


PLT_CN,forest_type,stand_age_yrs
<dbl>,<dbl>,<dbl>
1.279065e+15,141,26
1.279065e+15,141,60
1.507839e+15,141,52
1.507839e+15,141,50
1.507839e+15,141,107
1.507839e+15,141,14
1.507839e+15,141,82
2.407253e+13,141,30
1.258397e+14,141,73


In [23]:
#Third dataset: Clean SC_TREE to include only live longleaf pines (SPCD == 121)
clean_sc_tree <- sc_tree %>%
  filter(
       SPCD == 121,
       COUNTYCD %in% c(15,19),
       STATUSCD == 1,
       INVYR >= 2005
  ) %>%
  select(PLT_CN, SPCD, HT, DIA) %>%
  rename( #rename columns, match to first dataset
    tree_species  = SPCD,
    height_in        = HT,
    diameter_in      = DIA
  )
  head(clean_sc_tree, 15)

PLT_CN,tree_species,height_in,diameter_in
<dbl>,<dbl>,<dbl>,<dbl>
2.407160e+13,121,60,8.8
2.407258e+13,121,15,2.1
2.407253e+13,121,52,9.0
2.407253e+13,121,43,5.9
2.407253e+13,121,56,9.4
2.407253e+13,121,46,7.4
2.407253e+13,121,47,8.5
2.407253e+13,121,52,8.0
2.407253e+13,121,45,7.6


In [25]:
#Fourth dataset: Clean SC_SEED to include only live longleaf pines (SPCD == 121) in target counties
clean_sc_seedling <- sc_seedling %>%
  filter(
       SPCD == 121,
       COUNTYCD %in% c(15,19),
       INVYR >= 2005
  ) %>%
  select(PLT_CN, SPCD, TREECOUNT) %>%
  rename( #rename columns, match to first dataset
    seedling_ct   =TREECOUNT,
    seedling_species = SPCD
  )
  head(clean_sc_seedling, 15)

PLT_CN,seedling_species,seedling_ct
<dbl>,<dbl>,<dbl>
2.407258e+13,121,7
2.407253e+13,121,1
7.569276e+13,121,2
7.569276e+13,121,1
7.569276e+13,121,1
7.569276e+13,121,1
1.258397e+14,121,2
1.736746e+14,121,2
1.467769e+14,121,2


In [28]:
# Join and sort: Keep longleaf stands, then add trees, then add seedligns
clean_sc_longleaf <- clean_sc_plot %>%
  inner_join(clean_sc_type,     by = c("CN" = "PLT_CN")) %>%
  left_join(clean_sc_tree,      by = c("CN" = "PLT_CN")) %>%
  left_join(clean_sc_seedling,  by = c("CN" = "PLT_CN"))
head(clean_ll_stands, 20)

Warning message in left_join(., clean_sc_tree, by = c(CN = "PLT_CN")):
“Detected an unexpected many-to-many relationship between `x` and `y`.
ℹ Row 1 of `x` matches multiple rows in `y`.
ℹ Row 993 of `y` matches multiple rows in `x`.
ℹ If a many-to-many relationship is expected, set `relationship =
  "many-to-many"` to silence this warning.”
Warning message in left_join(., clean_sc_seedling, by = c(CN = "PLT_CN")):
“Detected an unexpected many-to-many relationship between `x` and `y`.
ℹ Row 30 of `x` matches multiple rows in `y`.
ℹ Row 2 of `y` matches multiple rows in `x`.
ℹ If a many-to-many relationship is expected, set `relationship =
  "many-to-many"` to silence this warning.”


countyid,inventory_yr,CN,PLOT,forest_type,stand_age_yrs,tree_species,height_in,diameter_in,seedling_species,seedling_ct
<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>
15,2005,2.407253e+13,162,141,30,121,52,9.0,121,1
15,2005,2.407253e+13,162,141,30,121,43,5.9,121,1
15,2005,2.407253e+13,162,141,30,121,56,9.4,121,1
15,2005,2.407253e+13,162,141,30,121,46,7.4,121,1
15,2005,2.407253e+13,162,141,30,121,47,8.5,121,1
15,2005,2.407253e+13,162,141,30,121,52,8.0,121,1
15,2005,2.407253e+13,162,141,30,121,45,7.6,121,1
15,2005,2.407253e+13,162,141,30,121,51,7.6,121,1
15,2005,2.407253e+13,162,141,30,121,48,6.3,121,1


In [29]:
write_csv(clean_sc_longleaf, "clean_data/clean_sc_longleaf.csv")

# Check: the file should now be in clean_data/
list.files("clean_data")

[1] "clean_sc_longleaf.csv"